# Clasificacion supervisada con Bank Marketing

En esta unidad estudiaremos un problema clasico de **clasificacion supervisada** usando Python, pandas y scikit-learn. Trabajaremos con el dataset **Bank Marketing** de UCI, cuyo objetivo es predecir si una persona aceptara o no una oferta de deposito a plazo despues de una campana de marketing telefonico.

La idea central es aprender un modelo a partir de ejemplos historicos. Cada fila del dataset contiene informacion conocida de una persona y de la campana; la respuesta `y` indica si esa persona contrato el producto (`yes`) o no (`no`). Como la respuesta tiene dos clases posibles, este es un problema de **clasificacion binaria**.

## 1. Introduccion

En aprendizaje supervisado tenemos datos de entrada y una respuesta conocida. El modelo observa muchos pares del tipo:

$$\text{entradas} \rightarrow \text{respuesta conocida}$$

Luego intenta aprender una regla que permita estimar la respuesta para casos nuevos.

La diferencia entre **clasificacion** y **regresion** esta en el tipo de respuesta que queremos predecir:

- En **regresion**, la salida es numerica continua. Por ejemplo: predecir el precio de una vivienda.
- En **clasificacion**, la salida es una categoria. Por ejemplo: predecir si un correo es spam o no spam.

En el caso del banco, no queremos predecir un monto continuo. Queremos responder una pregunta de tipo si/no: **esta persona aceptara el producto?**

## 2. Definicion del problema

Formalmente, nuestro problema queda definido asi:

- **Entrada**: variables sobre la persona, su historial y el contacto realizado por el banco.
- **Salida**: la variable objetivo `y`.
- **Clase positiva**: `yes`, es decir, la persona contrato el deposito a plazo.
- **Clase negativa**: `no`, es decir, la persona no contrato el producto.

Como la variable objetivo solo tiene dos valores posibles, trabajaremos con **clasificacion binaria**. En codigo convertiremos `yes` a `1` y `no` a `0`, porque muchas metricas y modelos esperan etiquetas numericas.

## 3. Dataset

Usaremos el dataset **Bank Marketing** del repositorio UCI Machine Learning Repository. Este dataset contiene registros de campanas de marketing directo de una institucion bancaria portuguesa.

El dataset mezcla distintos tipos de variables:

- **Variables numericas**: por ejemplo edad, balance o cantidad de contactos previos.
- **Variables categoricas**: por ejemplo trabajo, estado civil, educacion o tipo de contacto.
- **Variable objetivo**: `y`, que indica si la persona contrato el deposito a plazo.

Una advertencia importante: la variable `duration` mide cuanto duro la llamada. Esa informacion solo se conoce despues de realizar el contacto, por lo que usarla para predecir antes de llamar produciria **data leakage**. La eliminaremos antes de entrenar los modelos.

## 4. Carga del dataset

Primero importamos las bibliotecas necesarias. Si `ucimlrepo` no esta instalado en el entorno, la celda intenta instalarlo usando `%pip install ucimlrepo`. En el repositorio tambien declaramos esta dependencia en `pyproject.toml`, por lo que en el Dev Container deberia estar disponible despues de sincronizar el entorno.

In [ ]:
import importlib.util

if importlib.util.find_spec("ucimlrepo") is None:
    print("Instalando ucimlrepo en el kernel actual...")
    %pip install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from ucimlrepo import fetch_ucirepo

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
plt.style.use("default")
pd.set_option("display.max_columns", 80)

In [ ]:
bank_marketing = fetch_ucirepo(id=222)

X_original = bank_marketing.data.features.copy()
y_original = bank_marketing.data.targets.copy()

df = pd.concat([X_original, y_original], axis=1)
variable_objetivo = y_original.columns[0]

print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]:,}")
print(f"Variable objetivo: {variable_objetivo}")

In [ ]:
metadata_variables = bank_marketing.variables
display(metadata_variables)

## 5. Inspeccion inicial

Antes de entrenar modelos, conviene mirar los datos con calma. En esta etapa buscamos responder preguntas simples:

- Como se ven las primeras filas?
- Que tipos de datos tiene cada columna?
- Hay clases desbalanceadas?
- Hay valores faltantes o duplicados?

Estas preguntas parecen basicas, pero evitan muchos errores posteriores.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
conteo_clases = df[variable_objetivo].value_counts(dropna=False)
proporcion_clases = df[variable_objetivo].value_counts(normalize=True, dropna=False)

resumen_clases = pd.DataFrame({
    "n": conteo_clases,
    "proporcion": proporcion_clases,
})
display(resumen_clases)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
resumen_clases["n"].plot(kind="bar", ax=ax, color=["#4C78A8", "#F58518"])
ax.set_title("Distribucion de la variable objetivo")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de observaciones")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
valores_faltantes = df.isna().sum().sort_values(ascending=False)
valores_faltantes = valores_faltantes[valores_faltantes > 0]

print(f"Columnas con valores faltantes reales: {len(valores_faltantes)}")
display(valores_faltantes.to_frame("faltantes"))

duplicados = df.duplicated().sum()
print(f"Filas duplicadas exactas: {duplicados:,}")

## 6. Preparacion de datos

La preparacion tiene varios pasos. Primero hacemos una limpieza basica. Luego separamos entradas (`X`) y salida (`y`). Despues eliminamos `duration` para evitar data leakage.

Tambien necesitamos tratar de forma distinta las variables numericas y categoricas:

- A las variables numericas les aplicaremos imputacion y escalamiento.
- A las variables categoricas les aplicaremos imputacion y one-hot encoding.

Todo esto se integrara mas adelante dentro de un `Pipeline`, para que el mismo proceso se use durante entrenamiento y evaluacion.

In [ ]:
df_limpio = df.copy()

columnas_texto = df_limpio.select_dtypes(include="object").columns
for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].astype("object").map(
        lambda valor: valor.strip().lower() if isinstance(valor, str) else valor
    )

df_limpio = df_limpio.replace({"unknown": np.nan, "": np.nan})
filas_antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates().reset_index(drop=True)
filas_despues = len(df_limpio)

print(f"Filas antes de eliminar duplicados exactos: {filas_antes:,}")
print(f"Filas despues de eliminar duplicados exactos: {filas_despues:,}")
print(f"Filas eliminadas: {filas_antes - filas_despues:,}")

In [ ]:
mapeo_objetivo = {"no": 0, "yes": 1}
y = df_limpio[variable_objetivo].map(mapeo_objetivo).astype(int)
X = df_limpio.drop(columns=[variable_objetivo])

if "duration" in X.columns:
    X = X.drop(columns=["duration"])
    print("Se elimino la columna 'duration' para evitar data leakage.")
else:
    print("No se encontro la columna 'duration'.")

variables_numericas = X.select_dtypes(include=["number"]).columns.tolist()
variables_categoricas = X.select_dtypes(exclude=["number"]).columns.tolist()

print(f"Variables numericas ({len(variables_numericas)}): {variables_numericas}")
print(f"Variables categoricas ({len(variables_categoricas)}): {variables_categoricas}")

## 7. Division del dataset

Separaremos los datos en tres subconjuntos:

- **Train (60%)**: se usa para ajustar los modelos y hacer validacion cruzada.
- **Validation (20%)**: se usa para comparar modelos entrenados.
- **Test (20%)**: se reserva hasta el final para estimar desempeno en datos no usados durante la seleccion.

Usaremos `stratify` para conservar una proporcion similar de clases en cada subconjunto. Esto es importante cuando una clase es mucho menos frecuente que la otra.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

resumen_split = pd.DataFrame({
    "filas": [len(X_train), len(X_val), len(X_test)],
    "proporcion": [len(X_train) / len(X), len(X_val) / len(X), len(X_test) / len(X)],
    "tasa_clase_positiva": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(resumen_split)

## 8. Modelos

Entrenaremos tres modelos:

- **DummyClassifier**: baseline simple. Sirve para saber si un modelo real esta aportando valor.
- **LogisticRegression**: modelo lineal clasico para clasificacion binaria.
- **RandomForestClassifier**: ensamble de arboles de decision, capaz de capturar relaciones no lineales.

El baseline es importante porque un dataset desbalanceado puede producir una accuracy alta incluso con un modelo poco util. Si la mayoria de las personas responde `no`, un modelo que siempre predice `no` puede parecer bueno por accuracy, aunque no detecte clientes interesados.

In [ ]:
preprocesamiento_numerico = Pipeline(steps=[
    ("imputacion", SimpleImputer(strategy="median")),
    ("escalamiento", StandardScaler()),
])

preprocesamiento_categorico = Pipeline(steps=[
    ("imputacion", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocesamiento = ColumnTransformer(
    transformers=[
        ("num", preprocesamiento_numerico, variables_numericas),
        ("cat", preprocesamiento_categorico, variables_categoricas),
    ]
)

## 9. Entrenamiento con Pipeline, GridSearchCV y StratifiedKFold

`Pipeline` nos permite unir preparacion de datos y modelo en un solo objeto. Esto reduce errores, porque evita preparar train y test de formas distintas.

`GridSearchCV` prueba combinaciones de hiperparametros. Usaremos `StratifiedKFold` para que cada particion de validacion cruzada conserve la proporcion de clases. Como criterio de busqueda usaremos `average_precision`, una metrica util cuando la clase positiva es menos frecuente.

In [ ]:
modelos = {
    "dummy": {
        "estimador": DummyClassifier(random_state=RANDOM_STATE),
        "parametros": {
            "modelo__strategy": ["most_frequent", "stratified"],
        },
    },
    "regresion_logistica": {
        "estimador": LogisticRegression(
            max_iter=1000,
            solver="liblinear",
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "parametros": {
            "modelo__C": [0.1, 1.0, 10.0],
        },
    },
    "random_forest": {
        "estimador": RandomForestClassifier(
            n_estimators=120,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
        "parametros": {
            "modelo__max_depth": [8, None],
            "modelo__min_samples_leaf": [1, 5],
        },
    },
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
def construir_pipeline(estimador):
    return Pipeline(steps=[
        ("preprocesamiento", preprocesamiento),
        ("modelo", estimador),
    ])


def obtener_scores_positivos(modelo_entrenado, X_datos):
    indice_clase_positiva = list(modelo_entrenado.classes_).index(1)
    return modelo_entrenado.predict_proba(X_datos)[:, indice_clase_positiva]


def calcular_metricas(nombre, modelo_entrenado, X_datos, y_real, threshold=0.5):
    scores = obtener_scores_positivos(modelo_entrenado, X_datos)
    y_pred = (scores >= threshold).astype(int)
    return {
        "modelo": nombre,
        "threshold": threshold,
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred, zero_division=0),
        "recall": recall_score(y_real, y_pred, zero_division=0),
        "f1": f1_score(y_real, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_real, scores),
        "average_precision": average_precision_score(y_real, scores),
    }

In [ ]:
busquedas = {}
mejores_modelos = {}

for nombre, configuracion in modelos.items():
    print(f"Entrenando {nombre}...")
    pipeline = construir_pipeline(configuracion["estimador"])
    busqueda = GridSearchCV(
        estimator=pipeline,
        param_grid=configuracion["parametros"],
        scoring="average_precision",
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    busqueda.fit(X_train, y_train)
    busquedas[nombre] = busqueda
    mejores_modelos[nombre] = busqueda.best_estimator_
    print(f"  Mejor average precision CV: {busqueda.best_score_:.3f}")
    print(f"  Mejores parametros: {busqueda.best_params_}")

## 10. Metricas de evaluacion

La **matriz de confusion** separa las predicciones en cuatro casos:

- **TP**: verdaderos positivos. El modelo predijo `yes` y la respuesta real era `yes`.
- **TN**: verdaderos negativos. El modelo predijo `no` y la respuesta real era `no`.
- **FP**: falsos positivos. El modelo predijo `yes`, pero la respuesta real era `no`.
- **FN**: falsos negativos. El modelo predijo `no`, pero la respuesta real era `yes`.

Las metricas principales son:

$$\text{Accuracy} = \frac{TP + TN}{TP + FP + FN + TN}$$

$$\text{Precision} = \frac{TP}{TP + FP}$$

$$\text{Recall} = \frac{TP}{TP + FN}$$

$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

Tambien usaremos **ROC AUC** y **Average Precision**. Estas metricas trabajan con scores o probabilidades, no solo con una clase final. Por eso son utiles para comparar modelos antes de elegir un umbral de decision.

## 11. Evaluacion en validation

Ahora evaluamos los mejores modelos encontrados por `GridSearchCV` usando el conjunto de validacion. En esta primera comparacion usamos el umbral clasico `0.5`: si la probabilidad estimada de `yes` es al menos 0.5, predecimos clase positiva.

In [ ]:
metricas_validacion = [
    calcular_metricas(nombre, modelo, X_val, y_val, threshold=0.5)
    for nombre, modelo in mejores_modelos.items()
]

tabla_validacion = (
    pd.DataFrame(metricas_validacion)
    .set_index("modelo")
    .sort_values("average_precision", ascending=False)
)

display(tabla_validacion.style.format("{:.3f}"))

In [ ]:
criterio_seleccion = "average_precision"
nombre_mejor_modelo = tabla_validacion[criterio_seleccion].idxmax()
mejor_modelo = mejores_modelos[nombre_mejor_modelo]

print(f"Mejor modelo segun {criterio_seleccion}: {nombre_mejor_modelo}")
print("Parametros seleccionados:")
print(busquedas[nombre_mejor_modelo].best_params_)

In [ ]:
scores_val = obtener_scores_positivos(mejor_modelo, X_val)
pred_val_05 = (scores_val >= 0.5).astype(int)

print(classification_report(y_val, pred_val_05, target_names=["no", "yes"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_val,
    pred_val_05,
    display_labels=["no", "yes"],
    cmap="Blues",
    values_format="d",
    ax=ax,
)
ax.set_title(f"Matriz de confusion en validation\n{nombre_mejor_modelo}, threshold=0.5")
plt.tight_layout()
plt.show()

## 12. Threshold tuning

Muchos clasificadores no producen directamente una decision, sino un **score** o una probabilidad. El **threshold** es el punto de corte que transforma esa probabilidad en una clase.

Con threshold `0.5`, una observacion se clasifica como `yes` si su probabilidad estimada es al menos 50%. Pero ese valor no siempre es el mejor. Si bajamos el threshold, el modelo predice mas positivos: suele subir el recall, pero puede bajar la precision. Si subimos el threshold, el modelo exige mas evidencia para predecir `yes`: suele subir la precision, pero puede bajar el recall.

Aqui buscaremos el threshold que maximiza F1 en el conjunto de validacion.

In [ ]:
precision_curve, recall_curve, thresholds = precision_recall_curve(y_val, scores_val)

precision_threshold = precision_curve[:-1]
recall_threshold = recall_curve[:-1]
f1_threshold = 2 * (precision_threshold * recall_threshold) / (
    precision_threshold + recall_threshold + 1e-12
)

indice_mejor_threshold = int(np.nanargmax(f1_threshold))
threshold_optimo = float(thresholds[indice_mejor_threshold])

print(f"Threshold que maximiza F1 en validation: {threshold_optimo:.3f}")
print(f"Precision: {precision_threshold[indice_mejor_threshold]:.3f}")
print(f"Recall: {recall_threshold[indice_mejor_threshold]:.3f}")
print(f"F1: {f1_threshold[indice_mejor_threshold]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, precision_threshold, label="precision")
ax.plot(thresholds, recall_threshold, label="recall")
ax.plot(thresholds, f1_threshold, label="F1")
ax.axvline(threshold_optimo, color="black", linestyle="--", label="threshold optimo")
ax.set_title("Precision, recall y F1 segun threshold")
ax.set_xlabel("Threshold")
ax.set_ylabel("Valor de la metrica")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

## 13. Evaluacion final en test

El conjunto de test se usa una sola vez al final. Esto nos da una estimacion mas honesta del desempeno esperado, porque test no participo en el entrenamiento, en la busqueda de hiperparametros ni en la seleccion del threshold.

Compararemos el mejor modelo con threshold `0.5` y con el threshold ajustado en validacion.

In [ ]:
metricas_test_05 = calcular_metricas(
    f"{nombre_mejor_modelo}_threshold_0.5",
    mejor_modelo,
    X_test,
    y_test,
    threshold=0.5,
)

metricas_test_optimo = calcular_metricas(
    f"{nombre_mejor_modelo}_threshold_ajustado",
    mejor_modelo,
    X_test,
    y_test,
    threshold=threshold_optimo,
)

tabla_test = pd.DataFrame([metricas_test_05, metricas_test_optimo]).set_index("modelo")
display(tabla_test.style.format("{:.3f}"))

In [ ]:
scores_test = obtener_scores_positivos(mejor_modelo, X_test)
pred_test_optimo = (scores_test >= threshold_optimo).astype(int)

print(classification_report(y_test, pred_test_optimo, target_names=["no", "yes"], zero_division=0))

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_test_optimo,
    display_labels=["no", "yes"],
    cmap="Blues",
    values_format="d",
    ax=ax,
)
ax.set_title(f"Matriz de confusion en test\n{nombre_mejor_modelo}, threshold={threshold_optimo:.3f}")
plt.tight_layout()
plt.show()

## 14. Interpretacion del mejor modelo

La interpretacion depende del tipo de modelo seleccionado. En una regresion logistica podemos mirar coeficientes; en un random forest podemos mirar importancias de variables. Estas tecnicas no explican todo el comportamiento del modelo, pero ayudan a revisar que senales parecen estar influyendo mas en la prediccion.

In [ ]:
def extraer_importancias(modelo_entrenado, n=15):
    nombres_variables = modelo_entrenado.named_steps["preprocesamiento"].get_feature_names_out()
    estimador = modelo_entrenado.named_steps["modelo"]

    if hasattr(estimador, "feature_importances_"):
        valores = estimador.feature_importances_
        columna_valor = "importancia"
    elif hasattr(estimador, "coef_"):
        valores = estimador.coef_.ravel()
        columna_valor = "coeficiente"
    else:
        return pd.DataFrame()

    tabla = pd.DataFrame({
        "variable_transformada": nombres_variables,
        columna_valor: valores,
        "magnitud_absoluta": np.abs(valores),
    })
    return tabla.sort_values("magnitud_absoluta", ascending=False).head(n)


tabla_importancias = extraer_importancias(mejor_modelo)

if tabla_importancias.empty:
    print("El modelo seleccionado no expone coeficientes ni importancias de variables.")
else:
    display(tabla_importancias)

## 15. Analisis final

En esta unidad construimos un flujo completo de clasificacion supervisada: carga de datos, inspeccion, preparacion, division train/validation/test, entrenamiento con pipelines, busqueda de hiperparametros, evaluacion y ajuste de threshold.

El **DummyClassifier** funciona como baseline. Si un modelo real no supera claramente al baseline, probablemente no esta aprendiendo una senal util. En problemas desbalanceados, este punto es especialmente importante porque la accuracy puede ser enganosa: un modelo que predice casi siempre `no` puede obtener una accuracy aparentemente alta, pero detectar muy pocos clientes que realmente aceptarian la oferta.

La tension principal esta entre **precision** y **recall**. Si el banco quiere evitar contactar personas con baja probabilidad de aceptar, podria priorizar precision: menos falsos positivos, pero tambien menos oportunidades detectadas. Si el banco quiere no perder clientes potencialmente interesados, podria priorizar recall: menos falsos negativos, aunque con mas contactos que no terminaran en conversion.

Los **falsos positivos** representan personas que el modelo marca como interesadas, pero que no aceptan. Esto puede traducirse en costos operativos, llamadas innecesarias o mala experiencia del cliente. Los **falsos negativos** representan personas que si habrian aceptado, pero el modelo no identifica. Esto implica oportunidades comerciales perdidas.

Tambien vimos una limitacion importante: no todas las variables disponibles son validas para predecir antes de actuar. La columna `duration` fue eliminada porque solo se conoce despues de la llamada. Esta decision muestra que construir modelos no es solo aplicar algoritmos: tambien requiere entender el contexto, el momento en que cada dato esta disponible y el costo de cada tipo de error.

En una aplicacion real, el siguiente paso seria discutir con el area de negocio que error duele mas, revisar sesgos potenciales, monitorear desempeno en el tiempo y validar que el modelo mejore decisiones reales, no solo metricas offline.